# FB、LP–BP、FB–BP：显式检查两个边界
对应 `prop:lp-bp`、`prop:fb-bp`。

In [1]:
from pathlib import Path
import sys
root = Path.cwd()
if not (root / "qldpc_generator").is_dir():
    root = root.parent
sys.path.insert(0, str(root))
import numpy as np
from qldpc_generator.algebra import x, y


验证同一个量子比特置换 $P_1$ 同时满足 $$H_X^{\mathrm{target}}=P_0H_X^{\mathrm{source}}P_1^{\mathsf T},\qquad H_Z^{\mathrm{target}}=P_2H_Z^{\mathrm{source}}P_1^{\mathsf T}.\tag{1}$$ `p0,p1,p2` 保存旧索引到新索引的置换向量。

In [2]:
from qldpc_generator.algebra import symmetric_group_3,group_algebra_matrix
from qldpc_generator.equivalences import lp_to_bp,bp_to_lp,fb_to_bp,bp_to_fb
g=symmetric_group_3()
A=group_algebra_matrix(g,[[[0,1],[0]]])
B=group_algebra_matrix(g,[[[0,2]],[[1]]])
bp,eq=lp_to_bp(g,A,B)
assert eq.verify() and bp_to_lp(bp)[2].verify()
print('LP-BP on S3:',eq.verify(),'qubit permutation:',eq.p1.tolist())

LP-BP on S3: True qubit permutation: [24, 25, 26, 27, 28, 29, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]


LP–BP 使用 $[(\alpha,x),(\beta,y)]\mapsto(\alpha,\beta,yx)$。$S_3$ 测试会暴露将 $yx$ 误写成 $xy$ 的问题。

In [3]:
# F0 is fixed; F1 carries the natural three-point S3 action (not free).
a0=np.zeros((6,1),dtype=int); a1=np.array(g.labels)
bp,eq=fb_to_bp([[1,1]],[[1,1,1]],g,a0,a1,[[1,3]])
assert eq.verify() and bp_to_fb(bp)[2].verify()
print('FB-BP with nonfree fiber:',eq.verify())
try:
    bp_to_lp(bp)
except ValueError as exc:
    print('Expected refusal:',exc)
else:
    raise AssertionError('The LP converse requires free actions')

FB-BP with nonfree fiber: True
Expected refusal: This correspondence requires free actions on every chosen basis


FB→BP 不要求纤维作用自由；BP→LP 要求两因子的每个链基作用都自由。BP→FB 的反向函数另外检查所选基代表元下的单群标签覆盖条件。不满足时会报错，不会仅凭参数相同宣称对应。